In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv('Data/data_cleaned.csv', encoding="ISO-8859-1")

In [3]:
data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Transaction_Status
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,Completed
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,Completed
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,Completed
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,Completed
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,Completed


In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 399573 entries, 0 to 399572
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   InvoiceNo           399573 non-null  str    
 1   StockCode           399573 non-null  str    
 2   Description         399573 non-null  str    
 3   Quantity            399573 non-null  int64  
 4   InvoiceDate         399573 non-null  str    
 5   UnitPrice           399573 non-null  float64
 6   CustomerID          399573 non-null  float64
 7   Country             399573 non-null  str    
 8   Transaction_Status  399573 non-null  str    
dtypes: float64(2), int64(1), str(6)
memory usage: 27.4 MB


In [5]:
data = data[data['Transaction_Status'] == 'Completed']

In [6]:
cus_cluster = pd.read_csv('Data/customer_clusters.csv', encoding="ISO-8859-1")

In [7]:
cus_cluster.head()

,CustomerID,cluster
0,12347.0,2
1,12348.0,1
2,12350.0,1
3,12352.0,1
4,12353.0,1


In [8]:
data = pd.merge(data, cus_cluster[['CustomerID', 'cluster']], on='CustomerID', how='left')

In [9]:
data = data.dropna()

In [10]:
data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Transaction_Status,cluster
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/2010 8:34,1.69,13047.0,United Kingdom,Completed,2.0
10,536367,22745,POPPY'S PLAYHOUSE BEDROOM,6,12/1/2010 8:34,2.10,13047.0,United Kingdom,Completed,2.0
11,536367,22748,POPPY'S PLAYHOUSE KITCHEN,6,12/1/2010 8:34,2.10,13047.0,United Kingdom,Completed,2.0
12,536367,22749,FELTCRAFT PRINCESS CHARLOTTE DOLL,8,12/1/2010 8:34,3.75,13047.0,United Kingdom,Completed,2.0
13,536367,22310,IVORY KNITTED MUG COSY,6,12/1/2010 8:34,1.65,13047.0,United Kingdom,Completed,2.0


In [11]:
best_selling_products = data.groupby(['cluster', 'StockCode', 'Description'])['Quantity'].sum().reset_index()
best_selling_products = best_selling_products.sort_values(by=['cluster', 'Quantity'], ascending=[True, False])
top_products_per_cluster = best_selling_products.groupby('cluster').head(10)

In [12]:
customer_purchases = data.groupby(['CustomerID', 'cluster', 'StockCode'])['Quantity'].sum().reset_index()

In [13]:
recommendations = []
processed_customers = set()

for cluster in top_products_per_cluster['cluster'].unique():
    top_products = top_products_per_cluster[top_products_per_cluster['cluster'] == cluster]
    customers_in_cluster = data[data['cluster'] == cluster]['CustomerID'].unique()

    for customer in customers_in_cluster:
        if customer in processed_customers:
            continue

        processed_customers.add(customer)

        # Lấy danh sách sản phẩm khách hàng đã mua
        customer_purchased_products = customer_purchases[
            (customer_purchases['CustomerID'] == customer) & (customer_purchases['cluster'] == cluster)
        ]['StockCode'].tolist()

        # Tìm các sản phẩm tương tự dựa trên lịch sử mua hàng của khách hàng
        similar_products = data[data['StockCode'].isin(customer_purchased_products)]['Description'].unique()

        # Lọc các sản phẩm bán chạy nhất trong cụm, ưu tiên các sản phẩm tương tự
        recommended_products = top_products[
            top_products['Description'].isin(similar_products) & ~top_products['StockCode'].isin(customer_purchased_products)
        ]

        # Nếu không đủ sản phẩm tương tự, bổ sung từ danh sách bán chạy nhất
        num_recommendations_needed = 3 - len(recommended_products)
        if num_recommendations_needed > 0:
            additional_recommendations = top_products[
                ~top_products['StockCode'].isin(customer_purchased_products) & ~top_products['StockCode'].isin(recommended_products['StockCode'])
            ].head(num_recommendations_needed)
            recommended_products = pd.concat([recommended_products, additional_recommendations])

        recommendations.append(
            [customer, cluster] + recommended_products[['StockCode', 'Description']].values.flatten().tolist()
        )

In [14]:
recommendations_df = pd.DataFrame(recommendations, columns=['CustomerID', 'cluster', 'StockCode1', 'Description1', 'StockCode2', 'Description2', 'StockCode3', 'Description3'])

In [15]:
recommendations_df.sample(n=10)

,CustomerID,cluster,StockCode1,Description1,StockCode2,Description2,StockCode3,Description3
3691,14064.0,2.0,22616,PACK OF 12 LONDON TISSUES,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,85099B,JUMBO BAG RED RETROSPOT
3733,13196.0,2.0,22616,PACK OF 12 LONDON TISSUES,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,85099B,JUMBO BAG RED RETROSPOT
253,17430.0,1.0,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,84879,ASSORTED COLOUR BIRD ORNAMENT,15036,ASSORTED COLOURS SILK FAN
1749,14561.0,1.0,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,84879,ASSORTED COLOUR BIRD ORNAMENT,15036,ASSORTED COLOURS SILK FAN
3600,15382.0,2.0,85099B,JUMBO BAG RED RETROSPOT,84879,ASSORTED COLOUR BIRD ORNAMENT,85123A,WHITE HANGING HEART T-LIGHT HOLDER
4026,17350.0,2.0,22616,PACK OF 12 LONDON TISSUES,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,85099B,JUMBO BAG RED RETROSPOT
2983,18084.0,1.0,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,84879,ASSORTED COLOUR BIRD ORNAMENT,85123A,WHITE HANGING HEART T-LIGHT HOLDER
1264,16586.0,1.0,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,84879,ASSORTED COLOUR BIRD ORNAMENT,85123A,WHITE HANGING HEART T-LIGHT HOLDER
3231,14032.0,2.0,22616,PACK OF 12 LONDON TISSUES,85099B,JUMBO BAG RED RETROSPOT,21212,PACK OF 72 RETROSPOT CAKE CASES
1804,16739.0,1.0,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,84879,ASSORTED COLOUR BIRD ORNAMENT,85123A,WHITE HANGING HEART T-LIGHT HOLDER


In [16]:
recommendations_df.to_csv('Data/recommendations.csv', index=False)